In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az
import scoringrules as sr

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]

#splitting dataframe in preparation for lagging of variable data
pa_gs_yr_gsl = pa_gs[['year', 'growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy', 'station_id']]
pa_gs_non_gsl = pa_gs.drop(['growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy'], axis = 1)

#creating lagged dataset where variable data is paired with GSL
pa_gs_lagged = pa_gs_non_gsl.copy(deep=True)
pa_gs_lagged['year'] += 1
pa_gs_lagged_vars = pa_gs_lagged.merge(pa_gs_yr_gsl, on=["station_id", "year"], how = "left")

In [31]:
def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string
    
def bayesian_model_unstandardized(data, target_var_name, cov_list):

    cov_str = list_to_str(cov_list)
    cov_list_target_var = list(cov_list) + [target_var_name] + ['year', 'station_id']
    data = remove_nulls_simple(data, cov_list_target_var)
    
    formula = bmb.Formula(f"{target_var_name} ~ {cov_str} + year",
                      f"sigma ~ {cov_str} + year")
    model = bmb.Model(formula, data=data, dropna = True)
    output = model.fit(idata_kwargs={'log_likelihood':True})

    resid = data[target_var_name] - model.predict(output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
    
    return model, output, resid


all_covs_list = ['dtr_annual', 'dtr_spring', 'tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

cleaned_pa_gs_lagged = remove_nulls_simple(pa_gs_lagged_vars, all_covs_list + ['growing_season_length','year','station_id'])

In [19]:
lagged_model, lagged_output, lagged_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.141       31          111.76 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.137       31          126.36 draws/s       0:00:15    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.114       63          110.31 draws/s       0:00:18    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.170       31          121.24 draws/s       0:00:16    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 24 seconds.


In [6]:
dtr_list = ["dtr_annual"]
lagged_model_dtr_only, lagged_output_dtr_only, lagged_resid_dtr_only = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', dtr_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.826       3           708.47 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           1.344       3           907.52 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.977       3           949.65 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.692       7           896.08 draws/s       0:00:02    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 4 seconds.


In [7]:
lagged_lsf_model, lagged_lsf_output, lagged_lsf_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'last_spring_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.189       31          160.50 draws/s       0:00:12    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.185       31          121.85 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.177       31          168.77 draws/s       0:00:11    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.208       63          127.24 draws/s       0:00:15    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 22 seconds.


In [8]:
lagged_fff_model, lagged_fff_output, lagged_fff_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'first_fall_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.136       31          124.55 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.162       31          119.87 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.125       31          113.13 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.138       31          120.68 draws/s       0:00:16    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 24 seconds.


In [33]:
def get_prediction_ensembles(
    model,
    output,
    data,
    target_var_name,
    id_cols=("station_id", "year"),
    kind="pps",
    new_data=None,
    ci=0.95,
):
    """
    Build one row per input (station, year) with an ensemble of posterior predictive draws for that row.

    Parameters
    ----------
    model : The fitted Bambi model.
    output : The fitted result (model.fit output).
    data : The cleaned data used to fit the model (same row order).
    target_var_name : Name of the outcome column.
    id_cols :Columns that identify each row.
    kind : "pps" = predictions including observation noise or "response_params" = mean (mu) and sigma separately.
    new_data : Optional. Predict for new station/year rows instead of the training rows.
    ci : Width of the credible interval for summary columns.
    """

    # New data or existing data
    if new_data is not None:
        rows = new_data.reset_index(drop=True)
        pred_idata = model.predict(output, kind=kind, data=new_data, inplace=False)
    else:
        rows = data.reset_index(drop=True)
        pred_idata = model.predict(output, kind=kind, inplace=False)

    # Pulling predictive draws out of the InferenceData object.
    if kind == "pps":
        draws_da = pred_idata.posterior_predictive[target_var_name]
    else:
        draws_da = pred_idata.posterior["mu"]

    # Flatten the chain and draw dimensions into one "sample" dimension.
    obs_dim = [d for d in draws_da.dims if d not in ("chain", "draw")][0]
    stacked = draws_da.stack(sample=("chain", "draw")).transpose(obs_dim, "sample")
    arr = stacked.values  # shape: (n_rows, n_samples)

    # Checking length of input vs ensemble
    if arr.shape[0] != len(rows):
        raise ValueError(
            f"Got {arr.shape[0]} predicted rows but input has {len(rows)} rows."
        )

    # Summary statistics
    lower_q = (1 - ci) / 2 * 100
    upper_q = (1 + ci) / 2 * 100
    lower, median, upper = np.percentile(arr, [lower_q, 50, upper_q], axis=1)
    mean = arr.mean(axis=1)

    # Assemble results
    result = rows[list(id_cols)].copy()
    result["ensemble"] = list(arr)  # each cell holds a numpy array of draws
    result["n_draws"] = arr.shape[1]
    result["mean"] = mean
    result["median"] = median
    result[f"lower_{int(ci*100)}"] = lower
    result[f"upper_{int(ci*100)}"] = upper

    # Attach observed values when predicting on training data
    if new_data is None and target_var_name in rows.columns:
        result["observed"] = rows[target_var_name].values

    return result

In [37]:
lagged_gsl_predictions = lagged_model.predict(lagged_output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_gsl_predictions_series = pd.Series(lagged_gsl_predictions)

all_covs_w_gsl = all_covs_list + ['growing_season_length', 'station_id', 'year']
pa_gs_lagged_select_vars = remove_nulls_simple(pa_gs_lagged_vars, all_covs_w_gsl)
#pa_gs_lagged_no_na_gsl = pa_gs_lagged_no_na['growing_season_length']

pa_gs_lagged_w_preds = pd.concat([pa_gs_lagged_select_vars, lagged_gsl_predictions_series], axis=1)
pa_gs_lagged_w_preds.columns= [*pa_gs_lagged_w_preds.columns[:-1], 'gsl_preds']

pa_gs_gsl_obs = pa_gs_lagged_w_preds.pivot(index='year', columns='station_id', values = 'growing_season_length')

In [43]:
lagged_ensembles = get_prediction_ensembles(lagged_model, lagged_output, cleaned_pa_gs_lagged, "growing_season_length", kind="pps")
lagged_ensembles.head(15)

/home/reu/.venv/lib/python3.14/site-packages/bambi/models.py:925: FutureWarning: 'pps' has been replaced by 'response' and is not going to work in the future
  warnings.warn(


,station_id,year,ensemble,n_draws,mean,median,lower_95,upper_95,observed
0,USC00360022,2012,"[180.4350120411708, 188.6324065086945, 195.226...",4000,179.031277,179.399798,147.203243,210.574181,199.0
1,USC00360022,2013,"[183.7516455116963, 200.05739559182058, 178.47...",4000,196.027033,196.091165,168.872734,222.814365,166.0
2,USC00360022,2014,"[180.12033192073483, 164.28641182528352, 162.2...",4000,179.134879,179.347963,150.400555,207.402230,192.0
3,USC00360022,2015,"[176.3696031009574, 154.44149496346736, 190.94...",4000,180.464776,180.250982,151.599989,210.218374,199.0
4,USC00360022,2016,"[173.08594938698437, 172.0709304125354, 192.45...",4000,195.104735,195.209577,168.157032,222.289566,212.0
5,USC00360022,2017,"[178.34271797571873, 178.53899265732295, 206.0...",4000,192.870810,192.799852,165.553873,220.474342,176.0
6,USC00360022,2018,"[177.5459753735636, 166.28922374325793, 177.11...",4000,185.173124,185.543248,154.647526,214.325045,171.0
7,USC00360022,2019,"[198.78438413419158, 207.21993274556164, 215.5...",4000,189.984823,189.858803,161.162889,219.025573,173.0
8,USC00360022,2020,"[154.817325959013, 175.09566176889226, 190.419...",4000,164.644820,164.433038,133.654495,195.844693,160.0
9,USC00360022,2021,"[166.8386177450054, 182.67625783876923, 196.40...",4000,180.352642,180.724477,155.173108,204.088585,192.0


In [55]:
pa_gs_gsl_ens = lagged_ensembles.pivot(index='year', columns = 'station_id', values = 'ensemble')
pa_gs_gsl_ens_array = pa_gs_gsl_ens.to_numpy()

In [56]:
pa_gs_gsl_obs_array = pa_gs_gsl_obs.to_numpy()

In [57]:
lagged_full_variogram_score = sr.variogram_score(pa_gs_gsl_ens_array,pa_gs_gsl_obs_array)

ValueError: Forecasts shape (64, 176) and observations shape (64, 176) are not compatible for broadcasting!